In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*사용 시간 예상: Eagle r3 프로세서 기준 35분 (참고: 이는 추정치이며, 실제 실행 시간은 다를 수 있습니다.)*

## 학습 결과

이 튜토리얼을 마치면 다음과 같은 성과를 기대할 수 있습니다:
- 다체(multi‑body) Pauli 문자열이 어떻게 고전 최적화 문제의 다항식 압축을 가능하게 하는지를 포함하여 Pauli 상관관계 인코딩(PCE)의 이론적 원리를 이해합니다.
- 근미래 양자 하드웨어에서 대규모 최적화 작업을 인코딩하고 풀기 위해 실제로 PCE를 구현합니다.

## 사전 요구사항

이 튜토리얼을 진행하기 전에 다음 주제에 익숙해지는 것을 권장합니다:
- [변분 양자 알고리즘](/learning/courses/variational-algorithm-design)
- [QAOA와 max-cut](/tutorials/quantum-approximate-optimization-algorithm)

## 배경

이 튜토리얼에서는 *Pauli 상관관계 인코딩* (PCE) [\[1\]](#references)을 소개합니다. PCE는 양자 계산에서 더 높은 효율로 최적화 문제를 Qubit에 인코딩하도록 설계된 접근 방식입니다. PCE는 최적화 문제의 고전 변수를 다체(multi-body) Pauli 행렬 상관관계에 매핑하여 문제의 공간 요구사항을 다항식적으로 압축합니다. PCE를 사용하면 인코딩에 필요한 qubit 수가 줄어들어, 제한된 qubit 자원을 가진 근미래 양자 장치에 특히 유리합니다. 또한, PCE가 본질적으로 바렌 플래토(barren plateau) 문제를 완화하며, 이 현상에 대해 초다항식적 내성을 갖는다는 것이 분석적으로 증명되었습니다. 이러한 내장 기능은 양자 최적화 솔버에서 전례 없는 성능을 가능하게 합니다.

### 개요

PCE 접근 방식은 아래 [\[1\]](#references)의 그림 1에 나타난 것처럼 세 가지 주요 단계로 구성됩니다:
1. 최적화 문제를 Pauli 상관관계 공간으로 인코딩합니다.
2. 양자-고전 최적화 솔버를 사용하여 문제를 풉니다.
3. 해를 원래 최적화 공간으로 디코딩합니다.
PCE 접근 방식은 Pauli 상관관계 행렬을 처리할 수 있는 모든 양자 최적화 솔버에 적용 가능합니다.
![PCE 개요.](../docs/images/tutorials/solving-maxcut-with-reduced-qubit-requirements-using-pauli-correlation-encoding/af2cb835-88db-4a3d-9c86-51424b1a4bd3.avif)
[\[1\]](#references)의 그림 1에서 [max-cut](/tutorials/quantum-approximate-optimization-algorithm) 문제를 PCE 접근 방식을 설명하는 예시로 사용합니다. $m=9$ 노드의 max-cut 문제는 Pauli 상관관계 공간으로 인코딩되어, $n=3$ qubit $(Q_1, Q_2, Q_3)$에 걸친 2체(two-body) Pauli 행렬 상관관계로서 최적화 문제를 상관관계 행렬로 나타냅니다 &mdash; 노드의 색상은 각 인코딩된 노드에 사용된 Pauli 문자열을 나타냅니다.
예를 들어, 이진 변수 $x_1$에 해당하는 노드 1은 $Z_1 \otimes Z_2 \otimes I_3$의 기댓값으로 인코딩되며, $x_8$은 $I_1 \otimes Y_2 \otimes Y_3$으로 인코딩됩니다.
이는 문제의 $m$개 변수를 $ n = O(m^{1/2})$ Qubit으로 압축하는 것에 해당합니다. 더 일반적으로, $k>1$인 $k$체 상관관계는 $k$차 다항식 압축을 가능하게 합니다. 선택된 Pauli 집합은 상호 가환(mutually-commuting) Pauli 문자열의 세 가지 부분 집합으로 구성되어 있어, 단 세 가지 측정 설정으로 모든 $m$개의 상관관계를 실험적으로 추정할 수 있습니다.

원래 max-cut 목적 함수를 모방하는 Pauli 기댓값의 손실 함수 $\mathcal{L}$이 구성됩니다. 그런 다음, [VQE(Variational Quantum Eigensolver)](/learning/courses/quantum-diagonalization-algorithms/vqe)와 같은 양자-고전 최적화 솔버를 사용하여 손실 함수를 최적화합니다.

최적화가 완료되면, 해는 원래 최적화 공간으로 디코딩되어 최적의 max-cut 해를 산출합니다.

## 요구사항

이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하세요:
- Qiskit SDK v1.0 이상, [시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함
- Qiskit Runtime v0.22 이상 (`pip install qiskit-ibm-runtime`)

## 설정

In [ ]:
from itertools import combinations

import numpy as np
import rustworkx as rx
import networkx as nx

from scipy.optimize import minimize, OptimizeResult

from qiskit.circuit.library import efficient_su2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session
from rustworkx.visualization import mpl_draw
from qiskit_aer import AerSimulator

In [2]:
def calc_cut_size(graph, partition0, partition1):
    """Calculate the cut size of the given partitions of the graph."""

    cut_size = 0
    for edge0, edge1 in graph.edge_list():
        if edge0 in partition0 and edge1 in partition1:
            cut_size += 1
        elif edge0 in partition1 and edge1 in partition0:
            cut_size += 1
    return cut_size

## 소규모 시뮬레이터 예제

In [3]:
service = QiskitRuntimeService()
real_backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=156
)
backend = AerSimulator.from_backend(real_backend)
print(f"We are using the {backend.name}")

We are using the aer_simulator_from(ibm_pittsburgh)


### Step 1: Map classical inputs to a quantum problem

#### The max-cut problem
The max-cut problem is a combinatorial optimization problem that is defined on a graph $G = (V, E)$, where $V$ is the set of vertices and $E$ is the set of edges. The goal is to partition the vertices into two sets, $S$ and $V \setminus S$, such that the number of edges between the two sets is maximized.
For the detailed description of the max-cut problem, please refer to the [Quantum approximate optimization algorithm](/docs/tutorials/quantum-approximate-optimization-algorithm) tutorial.
The max-cut problem is also used as an example in the [Advanced techniques for QAOA](/docs/tutorials/advanced-techniques-for-qaoa) tutorial.
In those tutorials, the QAOA algorithm is used to solve the max-cut problem.


#### Graph -> Hamiltonian
Let us first consider a random graph with 100 nodes.

In [4]:
num_nodes = 100  # Number of nodes in graph
seed = 42
graph = rx.undirected_gnp_random_graph(num_nodes, 0.1, seed=seed)
mpl_draw(graph)

<Image src="../docs/images/tutorials/pauli-correlation-encoding-for-qaoa/extracted-outputs/37edb718-2bab-49d7-ad66-5f2f67d2aeff-0.avif" alt="Output of the previous code cell" />

In [ ]:
nx_graph = nx.Graph()
nx_graph.add_nodes_from(range(num_nodes))

In [6]:
for edge in graph.edge_list():
    nx_graph.add_edge(edge[0], edge[1])

In [7]:
curr_cut_size, partition = nx.approximation.one_exchange(nx_graph, seed=1)
print(f"Initial cut size: {curr_cut_size}")

Initial cut size: 345


We encode the graph with 100 nodes into two-body Pauli-matrix correlations across nine qubits (see the explanation below). The graph is represented as a correlation matrix, where each node is encoded by a Pauli string. The sign of the expectation value of the Pauli string indicates the partition of the node. For example, node 0 is encoded by a Pauli string, $\prod_0 = I_{8} \otimes ... I_2 \otimes X_1 \otimes X_0$. The sign of the expectation value of this Pauli string indicates the partition of node 0. We define a *Pauli-correlation encoding* (PCE) relative to $\prod$ as

$$ x_i \coloneqq \textit{sgn}(\langle\prod_i \rangle), $$

where $x_i$ is the partition of node $i$ and $\langle \prod_i \rangle \coloneqq  \langle \psi |\prod_i| \psi \rangle $ is the expectation value of the Pauli string encoding node $i$ over a quantum state $|\psi \rangle$.

Now, let's encode the graph into a Hamiltonian using PCE.
We divide the nodes into three sets: $S_1$, $S_2$, and $S_3$.
Then, we encode the nodes in each set using the Pauli strings with $X$, $Y$, and $Z$, respectively.

We need to extract a relationship between the number of nodes and qubits that we will need to encode all the nodes. Using all possible permutations for the encoding yields to:

$$
m=3\binom{n}{k}.
$$

In this example we consider $k=2$, hence,

$$
m  = \frac{3}{2} n(n-1).
$$

Therefore, the number of qubits $n$ needed to express a certain number of nodes $m$ read as:

$$
n = \left\lceil \frac{1 + \sqrt{1 + \tfrac{8}{3}m}}{2} \right\rceil.
$$

*Note that the $\lceil \cdot \rceil$ symbol represents the ceiling function, which rounds any real number up to the next integer. This ensures that the number of qubits is an integer.*

In [13]:
num_qubits = int(np.ceil((1 + np.sqrt(1 + (8 / 3) * num_nodes)) / 2))

list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

print(f"Number of qubits: {num_qubits}")
print("List 1:", node_x)
print("List 2:", node_y)
print("List 3:", node_z)

Number of qubits: 9
List 1: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
List 2: [33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65]
List 3: [66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [14]:
def build_pauli_correlation_encoding(pauli, node_list, n, k=2):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]] = pauli, pauli
        pauli_correlation_encoding.append(("".join(paulis)[::-1], 1))

    hamiltonian = []
    for pauli, weight in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(pauli, weight)]))

    return hamiltonian


pauli_correlation_encoding_x = build_pauli_correlation_encoding(
    "X", node_x, num_qubits
)
pauli_correlation_encoding_y = build_pauli_correlation_encoding(
    "Y", node_y, num_qubits
)
pauli_correlation_encoding_z = build_pauli_correlation_encoding(
    "Z", node_z, num_qubits
)

100개의 노드를 가진 그래프를 9개의 Qubit에 걸친 2체 Pauli 행렬 상관관계로 인코딩합니다(아래 설명 참조). 그래프는 상관관계 행렬로 표현되며, 각 노드는 Pauli 문자열로 인코딩됩니다. Pauli 문자열의 기댓값의 부호가 해당 노드의 분할을 나타냅니다. 예를 들어, 노드 0은 Pauli 문자열 $\prod_0 = I_{8} \otimes ... I_2 \otimes X_1 \otimes X_0$으로 인코딩됩니다. 이 Pauli 문자열의 기댓값의 부호가 노드 0의 분할을 나타냅니다. $\prod$에 대한 *Pauli 상관관계 인코딩* (PCE)을 다음과 같이 정의합니다:

$$ x_i \coloneqq \textit{sgn}(\langle\prod_i \rangle), $$

여기서 $x_i$는 노드 $i$의 분할이고, $\langle \prod_i \rangle \coloneqq  \langle \psi |\prod_i| \psi \rangle $는 양자 상태 $|\psi \rangle$에 대한 노드 $i$를 인코딩하는 Pauli 문자열의 기댓값입니다.
이제 PCE를 사용하여 그래프를 해밀토니안으로 인코딩해 보겠습니다.
노드를 $S_1$, $S_2$, $S_3$ 세 집합으로 나눕니다.
그런 다음, 각 집합의 노드를 각각 $X$, $Y$, $Z$ Pauli 문자열을 사용하여 인코딩합니다.
모든 노드를 인코딩하는 데 필요한 노드 수와 qubit 수 사이의 관계를 구해야 합니다. 인코딩에 가능한 모든 순열을 사용하면 다음과 같습니다:

$$
m=3\binom{n}{k}.
$$

이 예제에서는 $k=2$를 사용하므로,

$$
m  = \frac{3}{2} n(n-1).
$$

따라서, 특정 노드 수 $m$을 표현하는 데 필요한 qubit 수 $n$은 다음과 같습니다:

$$
n = \left\lceil \frac{1 + \sqrt{1 + \tfrac{8}{3}m}}{2} \right\rceil.
$$

*$\lceil \cdot \rceil$ 기호는 임의의 실수를 다음 정수로 올림하는 천장 함수(ceiling function)를 나타냅니다. 이를 통해 qubit 수가 정수가 됩니다.*

In [15]:
# Build the quantum circuit
qc = efficient_su2(num_qubits, su2_gates=["ry", "rz"], reps=2)
qc.draw("mpl")

<Image src="../docs/images/tutorials/pauli-correlation-encoding-for-qaoa/extracted-outputs/035f6b4a-4de0-452a-b60f-7260f9e3103a-0.avif" alt="Output of the previous code cell" />

In [16]:
# Optimize the circuit

pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
qc = pm.run(qc)

#### Loss function
For the loss function $\mathcal{L}$, we use a relaxation of the max-cut objective function as described in [\[1\]](#references), which is defined as $\mathcal{V}(\mathbf{x}) \coloneqq \sum_{(i, j) \in E} W_{i, j}(1-x_i x_j)$. Here, $W_{i, j}$ denotes the weight of the edge $(i, j)$, and $x_i$ represents the partition of node $i$.
The loss function  $\mathcal{L}$ is given by:

$\mathcal{L}\coloneqq \sum_{(i, j) \in E} W_{i, j} \text{tanh} (\alpha \langle\prod_i \rangle) \text{tanh} (\alpha \langle\prod_j \rangle) + \mathcal{L}^{(\text{reg})},$

where the max-cut objective function is replaced by the smooth hyperbolic tangents of the expectation values of the Pauli strings encoding the nodes. The regularization term $\mathcal{L}^{(\text{reg})}$ and the rescaling factor $\alpha$, proportional to the number of qubits, are introduced to improve the solver's performance.

The regularization term is defined as:

$\mathcal{L}^{(\text{reg})}$ is defined as $\mathcal{L}^{(\text{reg})} \coloneqq \beta \nu \lbrack \frac{1}{m} \sum_{i \in V} \text{tanh} (\alpha \langle\prod_i \rangle)^2 \rbrack ^2$

where $\beta=1/2$, $\nu = |E|/2 + (m -1) /4$, $|E|$ is the number of edges, and $m$ is the number of nodes in the graph.

In [ ]:
def loss_func_estimator(x, ansatz, hamiltonian, estimator, graph):
    """
    Calculates the specified loss function for the given ansatz, Hamiltonian,
    and graph.

    The expectation values of each Pauli string in the Hamiltonian are first
    obtained by running the ansatz on the quantum backend. These
    expectation values are then passed through the nonlinear function
    tanh(alpha * prod_i). The loss function is
    subsequently computed from these transformed values.
    """
    job = estimator.run(
        [
            (ansatz, hamiltonian[0], x),
            (ansatz, hamiltonian[1], x),
            (ansatz, hamiltonian[2], x),
        ]
    )
    result = job.result()

    # calculate the loss function
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1

    loss = 0
    alpha = num_qubits
    for edge0, edge1 in graph.edge_list():
        loss += np.tanh(alpha * node_exp_map[edge0]) * np.tanh(
            alpha * node_exp_map[edge1]
        )

    regulation_term = 0
    for i in range(len(graph.nodes())):
        regulation_term += np.tanh(alpha * node_exp_map[i]) ** 2
    regulation_term = regulation_term / len(graph.nodes())
    regulation_term = regulation_term**2
    beta = 1 / 2
    v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4
    regulation_term = beta * v * regulation_term

    loss = loss + regulation_term

    global experiment_result
    print(f"Iter {len(experiment_result)}: {loss}")
    experiment_result.append({"loss": loss, "exp_map": node_exp_map})
    return loss

### 2단계: 양자 하드웨어 실행을 위한 문제 최적화

#### 양자 Circuit

여기서 상태 $|\psi \rangle$는 $\mathbf{\theta}$로 매개변수화되며, 변분 접근 방식을 사용하여 이 매개변수 $\mathbf{\theta}$를 최적화합니다.
이 튜토리얼에서는 표현력과 구현의 용이성으로 인해 변분 알고리즘의 앤사츠(ansatz)로 `efficient_su2`를 사용합니다.
또한, 이 튜토리얼에서 나중에 소개될 완화된 손실 함수도 사용합니다.
그 결과, 더 적은 Qubit과 얕은 Circuit 깊이로 대규모 문제를 다룰 수 있습니다.

In [18]:
pce = []
pce.append(
    [op.apply_layout(qc.layout) for op in pauli_correlation_encoding_x]
)
pce.append(
    [op.apply_layout(qc.layout) for op in pauli_correlation_encoding_y]
)
pce.append(
    [op.apply_layout(qc.layout) for op in pauli_correlation_encoding_z]
)

In [ ]:
max_iter = 50
counter = {"i": 0}
last_x = {"value": None}
last_fun = {"value": None}

with Session(backend=backend) as session:
    estimator = Estimator(mode=session)

    experiment_result = []

    def loss_func(x):
        last_x["value"] = x.copy()
        if counter["i"] + 1 > max_iter:
            return last_fun["value"]
        counter["i"] += 1
        val = loss_func_estimator(
            x, qc, [pce[0], pce[1], pce[2]], estimator, graph
        )
        last_fun["value"] = val
        return val

    np.random.seed(seed)
    initial_params = np.random.rand(qc.num_parameters)

    result = minimize(
        loss_func, initial_params, method="COBYLA", options={"rhobeg": 1.0}
    )

    if counter["i"] >= max_iter:
        result = OptimizeResult(
            message=f"Return from COBYLA because the objective function "
            f"has been evaluated {max_iter} times.",
            success=False,
            status=3,
            fun=last_fun["value"],
            x=last_x["value"],
            nfev=counter["i"],
        )

print(result)

Iter 0: 159.88755362682548
Iter 1: 113.46202580636677
Iter 2: 56.76494226400048
Iter 3: 32.63357946896002
Iter 4: 21.517837239610117
Iter 5: 30.96034960483569
Iter 6: 20.780475923938027
Iter 7: 24.54251816279811
Iter 8: 27.834486461763042
Iter 9: 16.705460776812693
Iter 10: 18.020587887236864
Iter 11: 12.252379762741352
Iter 12: 5.253885750886939
Iter 13: 6.985984759592262
Iter 14: 6.908717244584757
Iter 15: 12.915466016863858
Iter 16: 4.105776920457279
Iter 17: 11.707504530740305
Iter 18: 7.154360511076546
Iter 19: 10.3890865704735
Iter 20: 10.376147647857252
Iter 21: 2.533430195296697
Iter 22: 3.8612421907795462
Iter 23: 6.103735057461906
Iter 24: -1.1190368234312347
Iter 25: 6.125915279494738
Iter 26: 11.086280445482455
Iter 27: 10.102569882302827
Iter 28: -0.02664415648133822
Iter 29: 7.621887727398785
Iter 30: 5.967346615554497
Iter 31: 3.85345716014828
Iter 32: 4.5494846149011
Iter 33: 10.006668112637232
Iter 34: -3.1927138938527877
Iter 35: 2.8829882366285116
Iter 36: 3.31300875

#### 손실 함수
손실 함수 $\mathcal{L}$으로, [\[1\]](#references)에 설명된 max-cut 목적 함수의 완화된 형태를 사용합니다. 이는 $\mathcal{V}(\mathbf{x}) \coloneqq \sum_{(i, j) \in E} W_{i, j}(1-x_i x_j)$로 정의됩니다. 여기서 $W_{i, j}$는 간선 $(i, j)$의 가중치를 나타내고, $x_i$는 노드 $i$의 분할을 나타냅니다.
손실 함수 $\mathcal{L}$은 다음과 같이 정의됩니다:

$\mathcal{L}\coloneqq \sum_{(i, j) \in E} W_{i, j} \text{tanh} (\alpha \langle\prod_i \rangle) \text{tanh} (\alpha \langle\prod_j \rangle) + \mathcal{L}^{(\text{reg})},$

여기서 max-cut 목적 함수는 노드를 인코딩하는 Pauli 문자열의 기댓값에 대한 매끄러운 쌍곡선 탄젠트(hyperbolic tangent)로 대체됩니다. 정규화 항 $\mathcal{L}^{(\text{reg})}$와 qubit 수에 비례하는 재조정 인수 $\alpha$는 솔버의 성능을 향상시키기 위해 도입됩니다.

정규화 항은 다음과 같이 정의됩니다:

$\mathcal{L}^{(\text{reg})}$는 $\mathcal{L}^{(\text{reg})} \coloneqq \beta \nu \lbrack \frac{1}{m} \sum_{i \in V} \text{tanh} (\alpha \langle\prod_i \rangle)^2 \rbrack ^2$으로 정의됩니다.

여기서 $\beta=1/2$, $\nu = |E|/2 + (m -1) /4$이며, $|E|$는 간선의 수이고, $m$은 그래프의 노드 수입니다.

In [20]:
# Calculate the partitions based on the final expectation values
# If the expectation value is positive, the node belongs to partition 0 (par0)
# Otherwise, the node belongs to partition 1 (par1)
def get_partitions(experiment_result):
    par0, par1 = set(), set()
    best_index = min(
        range(len(experiment_result)),
        key=lambda i: experiment_result[i]["loss"],
    )
    for i in experiment_result[best_index]["exp_map"]:
        if experiment_result[best_index]["exp_map"][i] >= 0:
            par0.add(i)
        else:
            par1.add(i)
    return par0, par1, best_index


par0, par1, best_index = get_partitions(experiment_result)
print(par0, par1)

{0, 2, 3, 8, 9, 11, 12, 13, 17, 18, 20, 22, 23, 24, 25, 26, 27, 30, 35, 37, 38, 40, 43, 46, 48, 49, 50, 51, 53, 57, 61, 62, 63, 66, 67, 68, 70, 71, 74, 77, 81, 82, 83, 84, 87, 88, 94, 96, 99} {1, 4, 5, 6, 7, 10, 14, 15, 16, 19, 21, 28, 29, 31, 32, 33, 34, 36, 39, 41, 42, 44, 45, 47, 52, 54, 55, 56, 58, 59, 60, 64, 65, 69, 72, 73, 75, 76, 78, 79, 80, 85, 86, 89, 90, 91, 92, 93, 95, 97, 98}


### Step 3: Qiskit 프리미티브를 사용하여 실행
이 튜토리얼에서는 시연 목적으로 최적화 루프에 `max_iter=50`을 설정합니다. 반복 횟수를 늘리면 더 나은 결과를 기대할 수 있습니다.

In [21]:
cut_size = calc_cut_size(graph, par0, par1)
print(f"Cut size: {cut_size}")

Cut size: 268


Once the training is complete, we perform one round of single-bit swap search to improve the solution as a classical post-processing step.
In this process, we swap the partitions of two nodes and evaluate the cut size. If the cut size is improved, we keep the swap. We repeat this process for all possible pairs of nodes connected by an edge.

In [22]:
cur_bits = []

for i in experiment_result[best_index]["exp_map"]:
    if experiment_result[best_index]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)
print(cur_bits)

[1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1]


In [23]:
# Swap the partitions and calculate the cut size


def swap_partitions(graph, cur_bits):
    best_cut = 0
    best_bits = []
    for edge0, edge1 in graph.edge_list():
        swapped_bits = cur_bits.copy()
        swapped_bits[edge0], swapped_bits[edge1] = (
            swapped_bits[edge1],
            swapped_bits[edge0],
        )

        cur_partition = [set(), set()]
        for i, bit in enumerate(swapped_bits):
            if bit > 0:
                cur_partition[0].add(i)
            else:
                cur_partition[1].add(i)
        cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
        if best_cut < cut_size:
            best_cut = cut_size
            best_bits = swapped_bits
    return best_cut, best_bits


best_cut, best_bits = swap_partitions(graph, cur_bits)
print(best_cut, best_bits)

279 [1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1]


### Step 4: 결과를 후처리하여 원하는 고전적 형식으로 반환
노드의 파티션은 노드를 인코딩하는 Pauli 문자열의 기댓값 부호를 평가하여 결정됩니다.

In [ ]:
# -------------------------Step 1-------------------------

num_nodes = 1500  # Number of nodes in graph
graph = rx.undirected_gnp_random_graph(num_nodes, 0.1, seed=seed)
nx_graph = nx.Graph()
nx_graph.add_nodes_from(range(num_nodes))
for edge in graph.edge_list():
    nx_graph.add_edge(edge[0], edge[1])

num_qubits = int(np.ceil((1 + np.sqrt(1 + (8 / 3) * num_nodes)) / 2))

list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

pauli_correlation_encoding_x = build_pauli_correlation_encoding(
    "X", node_x, num_qubits
)
pauli_correlation_encoding_y = build_pauli_correlation_encoding(
    "Y", node_y, num_qubits
)
pauli_correlation_encoding_z = build_pauli_correlation_encoding(
    "Z", node_z, num_qubits
)
print(f"We are using {num_qubits} qubits")

# -------------------------Step 2-------------------------
backend = real_backend
print(f"We are using the {backend.name}")
qc = efficient_su2(num_qubits, ["ry", "rz"], reps=2)
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
qc = pm.run(qc)
# -------------------------Step 3-------------------------
pce = []
pce.append(
    [op.apply_layout(qc.layout) for op in pauli_correlation_encoding_x]
)
pce.append(
    [op.apply_layout(qc.layout) for op in pauli_correlation_encoding_y]
)
pce.append(
    [op.apply_layout(qc.layout) for op in pauli_correlation_encoding_z]
)

# Run the optimization using a session.
max_iter = 50
counter = {"i": 0}
with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
    estimator.options.environment.job_tags = ["TUT_PCEFQ"]
    experiment_result = []

    def loss_func(x):
        last_x["value"] = x.copy()
        if counter["i"] + 1 > max_iter:
            return last_fun["value"]
        counter["i"] += 1
        val = loss_func_estimator(
            x, qc, [pce[0], pce[1], pce[2]], estimator, graph
        )
        last_fun["value"] = val
        return val

    np.random.seed(seed)
    initial_params = np.random.rand(qc.num_parameters)
    result = minimize(
        loss_func, initial_params, method="COBYLA", options={"rhobeg": 1.0}
    )
    if counter["i"] >= max_iter:
        result = OptimizeResult(
            message="Return from COBYLA because the objective function "
            "has been evaluated {max_iter} times.",
            success=False,
            status=3,
            fun=last_fun["value"],
            x=last_x["value"],
            nfev=counter["i"],
        )
print(result)

# -------------------------Step 4-------------------------

par0, par1, best_index = get_partitions(experiment_result)
cut_size = calc_cut_size(graph, par0, par1)
print(f"Cut size: {cut_size}")

best_bits = []
cur_bits = []
for i in experiment_result[best_index]["exp_map"]:
    if experiment_result[best_index]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)
best_cut, best_bits = swap_partitions(graph, cur_bits)
# Print final solution

print(
    f"The best max-cut value achieved for a graph with {num_nodes} nodes "
    f"on {num_qubits} qubits is {best_cut}"
)
print(f"and the specific partition we obtained is {best_bits}")

We are using 33 qubits
We are using the ibm_pittsburgh
Iter 0: 57399.57543902076
Iter 1: 56458.787143794
Iter 2: 40778.45608998947
Iter 3: 35571.58511146131
Iter 4: 33861.6835761173
Iter 5: 39697.22637736274
Iter 6: 34984.77893767163
Iter 7: 32051.882157096858
Iter 8: 26134.153216063707
Iter 9: 24914.322627065787
Iter 10: 24030.21227315425
Iter 11: 23047.463945514
Iter 12: 22629.42866110748
Iter 13: 17374.859132614685
Iter 14: 18020.11637762458
Iter 15: 17924.7066364044
Iter 16: 15825.1992250984
Iter 17: 16553.346711978447
Iter 18: 12393.565736512377
Iter 19: 11994.021456089155
Iter 20: 11199.994322735669
Iter 21: 9624.895532927634
Iter 22: 9073.811130188606
Iter 23: 9836.721241931278
Iter 24: 10555.925186133794
Iter 25: 9179.1179493286
Iter 26: 8495.394826965305
Iter 27: 8913.688189840399
Iter 28: 7830.448471810181
Iter 29: 7757.430542422075
Iter 30: 6796.187594518731
Iter 31: 7307.985913766867
Iter 32: 7340.225833330675
Iter 33: 7064.731899380469
Iter 34: 7632.270657372515
Iter 35: 7

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Advanced techniques for QAOA](/docs/tutorials/advanced-techniques-for-qaoa)
- [Combine error mitigation options with the Estimator primitive](/docs/tutorials/combine-error-mitigation-techniques)
</Admonition>

## References

[1] Sciorilli, M., Borges, L., Patti, T. L., García-Martín, D., Camilo, G., Anandkumar, A., & Aolita, L. (2024). Towards large-scale quantum optimization solvers with few qubits. arXiv preprint [arXiv:2401.09421](https://arxiv.org/abs/2401.09421).

## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_8ANZAlsKSFf6DA2)

© IBM Corp. 2024-2026